# Hoja de Trabajo 7 — CC2017 Modelación y Simulación
**Grupo 4** · Hospital Regional del Altiplano (camas de UCI)

## Task 1
### Task 1.1 — Desempeño teórico de las dos variantes

In [1]:
import math

# ---------- Parámetros del grupo ----------
g   = 4                              # número de grupo
mu  = 0.25                           # tasa de servicio [1/día]  (estancia media 1/mu = 4 días)
lam = 3.0 + 0.2 * ((g - 1) % 5)      # tasa de llegadas [pacientes/día]
c   = 17 + ((g - 1) % 3)             # número de camas
a   = lam / mu                       # carga ofrecida (Erlangs)
rho = a / c                          # utilización por cama
k_off = math.ceil(c - a) + 3         # camas fuera de servicio en el cierre parcial
T_b = 7                              # duración del cierre [días]

print(f"Grupo g = {g}")
print(f"λ     = {lam:.2f} pacientes/día")
print(f"c     = {c} camas")
print(f"a     = λ/μ = {a:.2f} Erlangs")
print(f"ρ     = a/c = {rho:.6f}")
print(f"k_off = ceil(c - a) + 3 = ceil({c - a:.2f}) + 3 = {k_off} camas   (T_b = {T_b} días)")
assert rho < 1, "La variante con espera necesita ρ < 1 para ser estable"


Grupo g = 4
λ     = 3.60 pacientes/día
c     = 17 camas
a     = λ/μ = 14.40 Erlangs
ρ     = a/c = 0.847059
k_off = ceil(c - a) + 3 = ceil(2.60) + 3 = 6 camas   (T_b = 7 días)


In [2]:
def erlang_b(c, a):
    # Recurrencia: B(k) = a*B(k-1) / (k + a*B(k-1)),  B(0) = 1
    B = 1.0
    for k in range(1, c + 1):
        B = a * B / (k + a * B)
    return B

def erlang_b_directa(c, a):
    # Fórmula directa: (a^c/c!) / sum_{k=0}^{c} a^k/k!   (solo para validar la recurrencia)
    num = a**c / math.factorial(c)
    den = sum(a**k / math.factorial(k) for k in range(c + 1))
    return num / den

def erlang_c(c, a):
    # C(c,a) = B / (1 - rho*(1 - B)),  requiere rho = a/c < 1
    rho = a / c
    B = erlang_b(c, a)
    return B / (1 - rho * (1 - B))

def erlang_c_directa(c, a):
    # Fórmula directa de Erlang C (solo para validar)
    rho = a / c
    cola = a**c / math.factorial(c) / (1 - rho)
    return cola / (sum(a**k / math.factorial(k) for k in range(c)) + cola)

# Validación cruzada de las dos formas de cada fórmula
print(f"Erlang B  recurrencia = {erlang_b(c, a):.10f} | directa = {erlang_b_directa(c, a):.10f}")
print(f"Erlang C  vía B       = {erlang_c(c, a):.10f} | directa = {erlang_c_directa(c, a):.10f}")


Erlang B  recurrencia = 0.0967215320 | directa = 0.0967215320
Erlang C  vía B       = 0.4118087900 | directa = 0.4118087900


#### (a) Variante con espera — M/M/c

In [3]:
# ---------- Variante CON espera (M/M/c) ----------
C_ca = erlang_c(c, a)                          # P(esperar) = C(c,a)
Lq   = C_ca * rho / (1 - rho)                  # Lq = C * rho/(1-rho)
Wq_d = Lq / lam                                # Wq = Lq / λ            [días]
W_d  = Wq_d + 1 / mu                           # W  = Wq + 1/μ          [días]
L    = Lq + a                                  # L  = Lq + a  (a = camas ocupadas en promedio)

Wq_h = Wq_d * 24                               # a horas
W_h  = W_d * 24

# Verificación independiente de Wq con la forma cerrada  Wq = C / (c*μ - λ)
Wq_alt_d = C_ca / (c * mu - lam)

print("=== Variante con espera (M/M/c) ===")
print(f"a      = {a:.4f}")
print(f"ρ      = {rho:.6f}")
print(f"C(c,a) = {C_ca:.6f}   (P[esperar] ≈ {C_ca*100:.2f} %)")
print(f"Lq     = {Lq:.6f} pacientes")
print(f"Wq     = {Wq_d:.6f} días = {Wq_h:.4f} horas")
print(f"W      = {W_d:.6f} días = {W_h:.4f} horas")
print(f"L      = {L:.6f} pacientes")
print(f"(check Wq = C/(cμ-λ): {Wq_alt_d:.6f} días)")


=== Variante con espera (M/M/c) ===
a      = 14.4000
ρ      = 0.847059
C(c,a) = 0.411809   (P[esperar] ≈ 41.18 %)
Lq     = 2.280787 pacientes
Wq     = 0.633552 días = 15.2052 horas
W      = 4.633552 días = 111.2052 horas
L      = 16.680787 pacientes
(check Wq = C/(cμ-λ): 0.633552 días)


#### (b) Variante sin espera — M/M/c/c (Erlang B / sistema de pérdida)

In [4]:
# ---------- Variante SIN espera (M/M/c/c) ----------
B_ca    = erlang_b(c, a)                       # prob. de bloqueo B(c,a)
lam_ef  = lam * (1 - B_ca)                     # tasa efectiva de ingresos: solo entran los no bloqueados
L_sin   = lam_ef / mu                          # camas ocupadas promedio = λ_ef/μ = a(1-B)
desvio  = lam * B_ca                           # pacientes desviados por día
W_sin_d = 1 / mu                               # no hay espera: W = 1/μ
W_sin_h = W_sin_d * 24

print("=== Variante sin espera (M/M/c/c) ===")
print(f"B(c,a)          = {B_ca:.6f}   ({B_ca*100:.3f} % de pacientes desviados)")
print(f"λ_ef            = {lam_ef:.6f} pacientes/día   (λ = {lam:.2f})")
print(f"Camas ocupadas  = {L_sin:.6f}   (de {c}; ocupación por cama = {L_sin/c:.4f})")
print(f"Desvíos         = {desvio:.6f} pacientes/día")
print(f"W = 1/μ         = {W_sin_d:.2f} días = {W_sin_h:.1f} horas; Wq = 0")


=== Variante sin espera (M/M/c/c) ===
B(c,a)          = 0.096722   (9.672 % de pacientes desviados)
λ_ef            = 3.251802 pacientes/día   (λ = 3.60)
Camas ocupadas  = 13.007210   (de 17; ocupación por cama = 0.7651)
Desvíos         = 0.348198 pacientes/día
W = 1/μ         = 4.00 días = 96.0 horas; Wq = 0


#### (c) Consistencia con la ley de Little

$L = \lambda_{\text{ef}}\,W$ debe cumplirse **dentro de cada variante** (con $\lambda_{\text{ef}}$ la tasa a la que realmente entran pacientes al sistema). Además, la relación entre variantes y las identidades internas:

- **Con espera:** $\lambda_{\text{ef}} = \lambda$ (nadie se pierde), por lo que $L_q = \lambda W_q$ y $L = \lambda W$.
- **Sin espera:** $L = \lambda_{\text{ef}}\,W = \lambda(1-B)\cdot\dfrac{1}{\mu}$, y $L_q = \lambda_{\text{ef}} W_q = 0$ trivialmente.

In [5]:
# ---------- Verificación con la ley de Little: L = λ_ef * W ----------
# Variante con espera (λ_ef = λ)
little_Lq = lam * Wq_d            # debe igualar Lq
little_L  = lam * W_d             # debe igualar L

# Variante sin espera (λ_ef = λ(1-B))
little_L_sin = lam_ef * W_sin_d   # debe igualar L_sin

print("--- Variante con espera ---")
print(f"Lq      = {Lq:.8f}   | λ·Wq = {little_Lq:.8f}   | dif = {abs(Lq-little_Lq):.2e}")
print(f"L       = {L:.8f}   | λ·W  = {little_L:.8f}   | dif = {abs(L-little_L):.2e}")
print("--- Variante sin espera ---")
print(f"L       = {L_sin:.8f}   | λ_ef·W = {little_L_sin:.8f}   | dif = {abs(L_sin-little_L_sin):.2e}")

# Error que se cometería usando λ en lugar de λ_ef en la variante sin espera
L_mal = lam * W_sin_d             # = a  (ignora que los bloqueados nunca entran)
print("\n--- Error al usar λ en vez de λ_ef (sin espera) ---")
print(f"λ·W = {L_mal:.4f}  vs  L correcto = {L_sin:.4f}   → sobreestima L en {(L_mal/L_sin - 1)*100:.2f} % (factor 1/(1-B) = {1/(1-B_ca):.4f})")
assert abs(Lq-little_Lq) < 1e-9 and abs(L-little_L) < 1e-9 and abs(L_sin-little_L_sin) < 1e-9


--- Variante con espera ---
Lq      = 2.28078714   | λ·Wq = 2.28078714   | dif = 0.00e+00
L       = 16.68078714   | λ·W  = 16.68078714   | dif = 0.00e+00
--- Variante sin espera ---
L       = 13.00720994   | λ_ef·W = 13.00720994   | dif = 0.00e+00

--- Error al usar λ en vez de λ_ef (sin espera) ---
λ·W = 14.4000  vs  L correcto = 13.0072   → sobreestima L en 10.71 % (factor 1/(1-B) = 1.1071)


In [6]:
# ---------- Resumen Task 1.1 ----------
print(f"{'Parámetros del grupo 4':-^62}")
print(f"λ = {lam:.1f}/día | c = {c} | a = {a:.1f} | ρ = {rho:.4f} | k_off = {k_off}")
print(f"{'Variante con espera (M/M/c)':-^62}")
print(f"C(c,a)={C_ca:.4f}  Lq={Lq:.4f}  Wq={Wq_h:.3f} h  W={W_h:.3f} h  L={L:.4f}")
print(f"{'Variante sin espera (M/M/c/c)':-^62}")
print(f"B(c,a)={B_ca:.4f}  λ_ef={lam_ef:.4f}/día  camas ocupadas={L_sin:.4f}")


--------------------Parámetros del grupo 4--------------------
λ = 3.6/día | c = 17 | a = 14.4 | ρ = 0.8471 | k_off = 6
-----------------Variante con espera (M/M/c)------------------
C(c,a)=0.4118  Lq=2.2808  Wq=15.205 h  W=111.205 h  L=16.6808
----------------Variante sin espera (M/M/c/c)-----------------
B(c,a)=0.0967  λ_ef=3.2518/día  camas ocupadas=13.0072
